In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

# Câu 1
## a

In [ ]:
with open('Data/Chuong2/pages/page1.html', 'r', encoding='utf-8') as f:
    page1 = f.read()

In [ ]:
soup = BeautifulSoup(page1, 'html5lib')
soup

In [ ]:
product_cards = soup.find_all('article', class_='product-card')
for i in product_cards:
    print(i)

## b

In [ ]:
with open('Data/Chuong2/pages/page2.html', 'r', encoding='utf-8') as f:
    page2 = f.read()
    
soup2 = BeautifulSoup(page2, 'html5lib')

In [ ]:
data_c1 = []

In [ ]:
for i in product_cards:
    ten_sp = i.find('h3', class_='product-name').text.strip()
    gia_goc = i.find('span', class_='list-price').text.strip()
    gia_ban = i.find('span', class_='sale-price').text.strip()
    try:
        khuyen_mai = i.find('p', class_='promotion').text.strip()
    except:
        khuyen_mai=None
        
    data_c1.append({
        'ten_sp':ten_sp,
        'gia_goc':gia_goc,
        'gia_ban':gia_ban,
        'khuyen_mai':khuyen_mai
    })

In [ ]:
product_cards_2 = soup2.find_all('article', class_='product-card')

for i in product_cards_2:
    ten_sp = i.find('h3', class_='product-name').text.strip()
    gia_goc = i.find('span', class_='list-price').text.strip()
    gia_ban = i.find('span', class_='sale-price').text.strip()
    try:
        khuyen_mai = i.find('p', class_='promotion').text.strip()
    except:
        khuyen_mai=None
    data_c1.append({
        'ten_sp':ten_sp,
        'gia_goc':gia_goc,
        'gia_ban':gia_ban,
        'khuyen_mai':khuyen_mai
    })


In [ ]:
df_c1 = pd.DataFrame(data_c1, columns=['ten_sp', 'gia_goc', 'gia_ban', 'khuyen_mai'])
df_c1

In [ ]:
df_c1.to_csv('Cau1/dien_thoai_tho.csv', encoding='utf-8', index=False)

## c

In [ ]:
df = pd.read_csv('Cau1/dien_thoai_tho.csv')

df.rename(columns={
    'gia_goc': 'gia_goc_vnd',
    'gia_ban': 'gia_ban_vnd',
}, inplace=True)

In [ ]:
cols = ['gia_goc_vnd', 'gia_ban_vnd']

for col in cols:
    df[col] = df[col].str.replace(' đ', '').str.replace('.','').astype('int64')


In [ ]:
df['ty_le_giam'] = round((df['gia_goc_vnd'] - df['gia_ban_vnd'])/df['gia_goc_vnd'], 2)*100

In [ ]:
df.fillna({
    'khuyen_mai':'Không có khuyến mãi'
}, inplace=True)

In [ ]:
df

In [ ]:
df.to_csv('Cau1/dien_thoai_sach.csv', encoding='utf-8', index=False)

# Câu 2

## a

In [ ]:
df = pd.read_csv('Data/Chuong3/reviews_raw.csv')
df

In [ ]:
chuan_hoa = pd.read_csv('Data/Chuong3/tu_dien_chuan_hoa.csv')
chuan_hoa

In [ ]:
chuan_hoa_dict = chuan_hoa.set_index(chuan_hoa.columns[0]).to_dict()[chuan_hoa.columns[1]]
chuan_hoa_dict

In [ ]:
emoji_map = pd.read_csv('Data/Chuong3/emoji_map.csv')
emoji_map

In [ ]:
emoji_dict = emoji_map.set_index(emoji_map.columns[0]).to_dict()[emoji_map.columns[1]]
emoji_dict

In [ ]:
df['review'] = df['review'].str.lower()

In [ ]:
import re

def remove_html(text):
    return re.sub(r'<[^>]+>', '', text)

def remove_digits(text):
    return re.sub(r'\d+', '', text)

def remove_urls(text):
    tmp =  re.sub(r'https?://\S+', '', text)
    res = re.sub(r'www\.\S+', '', tmp)
    return res

def normalize_space(text):
    return re.sub(r'\s+', ' ', text)

df['review'] = df['review'].apply(remove_html)
df['review'] = df['review'].apply(remove_digits)
df['review'] = df['review'].apply(remove_urls)
df['review'] = df['review'].apply(normalize_space)

In [ ]:
resa = df.copy()
resa.rename(columns={
    'review':'review_buoc1'
}, inplace=True)

resa

In [ ]:
resa.to_csv('Cau2/buoc1.csv', encoding='utf-8', index=False)

## b

In [ ]:
def remove_punctuation(text):
    res = re.sub(r'[^\w\s]*', '', text)
    return res


def replace_words(text, dict):
    l = text.split()

    for i in range(len(l)):
        if l[i] in dict.keys():
            l[i] = dict[l[i]]
    
    return ' '.join(l)

resa['review_buoc1'] = resa['review_buoc1'].apply(remove_punctuation)
resa['review_buoc1'] = resa['review_buoc1'].apply(lambda x: replace_words(x, emoji_dict))
resa['review_buoc1'] = resa['review_buoc1'].apply(lambda x: replace_words(x, chuan_hoa_dict))
resa['review_buoc1'] = resa['review_buoc1'].apply(normalize_space)


In [ ]:
resb = resa.copy()

In [ ]:
resb.rename(columns={
    'review_buoc1':'review_buoc2'
}, inplace=True)

resb

In [ ]:
resb.to_csv('Cau2/buoc2.csv', encoding='utf-8', index=False)

## c

In [ ]:
import underthesea

In [ ]:
resc = resb.copy()

In [ ]:
resc

In [ ]:
def tokenize(text):
    return underthesea.word_tokenize(text, format='text')

def remove_stop_words(s):
    list = s.split()
    with open('Data/Chuong3/stopwords.txt', 'r', encoding='utf-8') as f:
        stop_words = set(f.read().split('\n'))
    
    tmp =  [i for i in list if i not in stop_words]
    return ' '.join(tmp)

In [ ]:
resc['tokens'] = resc['review_buoc2'].apply(tokenize)
resc['tokens'] = resc['tokens'].apply(remove_stop_words)

In [ ]:
resc.drop(['review_buoc2'], axis=1, inplace=True)

In [ ]:
resc

In [ ]:
resc.to_csv('Cau2/buoc3.csv', encoding='utf-8', index=False)

## d

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

all_words = [word for tokens in resc['tokens'].str.split() for word in tokens]

word_counts = Counter(all_words)
top10 = pd.DataFrame(word_counts.most_common(10), columns=['token', 'tan_suat'])

top10

In [ ]:
top10.to_csv('Cau2/tan_suat_top_10.csv', encoding='utf-8', index=False)

In [ ]:
resc

## e

In [ ]:
from wordcloud import WordCloud

wc = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='plasma'
).generate_from_frequencies(word_counts)


plt.figure(figsize=(12,6))
plt.imshow(wc, interpolation="bilinear")
plt.axis('off')
plt.title('Word cloud tổng thể')
plt.show()

wc.to_file('Cau2/wordcloud.png')

In [ ]:
plt.figure(figsize=(10,20))
sns.barplot(data=top10, x='tan_suat', y='token', hue='token')
plt.title('Top 20 từ vựng xuất hiện nhiều nhất')
plt.xlabel('Tần suất')
plt.ylabel('Từ vựng')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('Cau2/tan_suat_top10.png')
plt.show()